# Sri Lanka Weather Analytics - Weekly Maximum Temperature Analysis

This notebook identifies the hottest months based on average temperature_2m_max and calculates weekly maximum temperatures within those months.

**Requirements: 4.2, 4.3**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, max as spark_max, avg, month, year, weekofyear,
    to_date, dense_rank, round as spark_round, min as spark_min
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, StringType
)
import os

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("WeeklyMaxTemperatureAnalysis") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.session.timeZone", "Asia/Colombo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
# Define weather data schema
weather_schema = StructType([
    StructField("location_id", IntegerType(), nullable=False),
    StructField("date", StringType(), nullable=False),
    StructField("weather_code", IntegerType(), nullable=True),
    StructField("temperature_2m_max", FloatType(), nullable=True),
    StructField("temperature_2m_min", FloatType(), nullable=True),
    StructField("temperature_2m_mean", FloatType(), nullable=True),
    StructField("apparent_temperature_max", FloatType(), nullable=True),
    StructField("apparent_temperature_min", FloatType(), nullable=True),
    StructField("apparent_temperature_mean", FloatType(), nullable=True),
    StructField("daylight_duration", FloatType(), nullable=True),
    StructField("sunshine_duration", FloatType(), nullable=True),
    StructField("precipitation_sum", FloatType(), nullable=True),
    StructField("rain_sum", FloatType(), nullable=True),
    StructField("precipitation_hours", FloatType(), nullable=True),
    StructField("wind_speed_10m_max", FloatType(), nullable=True),
    StructField("wind_gusts_10m_max", FloatType(), nullable=True),
    StructField("wind_direction_10m_dominant", FloatType(), nullable=True),
    StructField("shortwave_radiation_sum", FloatType(), nullable=True),
    StructField("et0_fao_evapotranspiration", FloatType(), nullable=True),
    StructField("sunrise", StringType(), nullable=True),
    StructField("sunset", StringType(), nullable=True)
])

In [ ]:
# Load weather data
weather_path = "../dataset/weatherData.csv"

weather_df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "") \
    .option("nanValue", "NaN") \
    .schema(weather_schema) \
    .csv(weather_path)

# Parse date and extract year, month, week
weather_df = weather_df.withColumn("parsed_date", to_date(col("date"), "M/d/yyyy"))
weather_df = weather_df \
    .withColumn("year", year(col("parsed_date"))) \
    .withColumn("month", month(col("parsed_date"))) \
    .withColumn("week", weekofyear(col("parsed_date")))

print(f"Total records: {weather_df.count()}")

## Step 1: Identify Hottest Months

Calculate the average maximum temperature for each month across all years and districts, then identify the top 3 hottest months.

In [ ]:
# Filter out records with null temperature values
valid_temp_df = weather_df.filter(col("temperature_2m_max").isNotNull())

# Calculate average max temperature by month
monthly_avg = valid_temp_df.groupBy("month").agg(
    spark_round(avg("temperature_2m_max"), 2).alias("avg_max_temperature")
)

# Rank months by average max temperature
window_spec = Window.orderBy(col("avg_max_temperature").desc())
ranked_months = monthly_avg.withColumn("rank", dense_rank().over(window_spec))

# Get top 3 hottest months
top_n = 3
hottest_months_df = ranked_months.filter(col("rank") <= top_n) \
    .select("month", "avg_max_temperature", "rank") \
    .orderBy("rank")

print("Hottest Months by Average Maximum Temperature:")
hottest_months_df.show()

## Step 2: Calculate Weekly Maximum Temperatures

Filter data to only include records from the hottest months, then calculate the maximum temperature_2m_max for each week.

In [ ]:
# Get list of hottest month numbers
hottest_month_list = [row["month"] for row in hottest_months_df.collect()]
print(f"Hottest months: {hottest_month_list}")

# Filter weather data to only include hottest months
filtered_df = weather_df.filter(
    (col("month").isin(hottest_month_list)) & 
    (col("temperature_2m_max").isNotNull())
)

# Calculate weekly maximum temperatures grouped by year, month, and week
weekly_max_df = filtered_df.groupBy("year", "month", "week").agg(
    spark_round(spark_max("temperature_2m_max"), 2).alias("weekly_max_temperature")
)

# Order by year, month, week
weekly_max_df = weekly_max_df.orderBy("year", "month", "week")

print(f"Total weeks analyzed: {weekly_max_df.count()}")
weekly_max_df.show(30)

## Step 3: Summary Statistics

In [ ]:
# Calculate summary statistics
stats = weekly_max_df.agg(
    spark_max("weekly_max_temperature").alias("overall_max"),
    spark_round(avg("weekly_max_temperature"), 2).alias("overall_avg"),
    spark_min("year").alias("min_year"),
    spark_max("year").alias("max_year")
).collect()[0]

print(f"Year range: {stats['min_year']} - {stats['max_year']}")
print(f"Overall max temperature: {stats['overall_max']}°C")
print(f"Overall avg weekly max: {stats['overall_avg']}°C")

## Step 4: Yearly Breakdown

In [ ]:
# Yearly breakdown of weekly max temperatures
yearly_summary = weekly_max_df.groupBy("year").agg(
    spark_max("weekly_max_temperature").alias("max_temp"),
    spark_round(avg("weekly_max_temperature"), 2).alias("avg_temp")
).orderBy("year")

print("Yearly breakdown of weekly max temperatures:")
yearly_summary.show(20)

In [ ]:
# Stop Spark session
spark.stop()
print("Analysis complete!")